<div style="display:inline-block; padding:14px 30px; border-radius:40px 12px 40px 12px; background:#c7e9d0; color:#1e4d3a; font-family:'Segoe UI',sans-serif; font-size:1.6rem; font-weight:500; box-shadow:0 6px 18px rgba(0,0,0,0.06); border:1px solid rgba(255,255,255,0.4);">
    🌿 Fatemeh Habib Zadeh
</div>

In [3]:
import numpy as np
import pandas as pd

In [4]:
#خواندن دیتاست Train
train_df = pd.read_csv("auto-train-preprocessed.csv")

print("Shape:", train_df.shape)
print(train_df.head())

Shape: (352, 2)
   HorsePower       MPG
0    0.038043  0.638298
1    0.646739  0.106383
2    0.157609  0.398936
3    0.239130  0.239362
4    0.728261  0.186170


In [5]:
# جدا کردن ورودی و خروجی

# ورودی مدل
X_train = train_df["HorsePower"].values.reshape(-1, 1)   # shape: (n, 1)
# مقدار هدفی است که مدل باید پیش‌بینی کند
y_train = train_df["MPG"].values.reshape(-1, 1)           # shape: (n, 1)

print("X shape:", X_train.shape)
print("y shape:", y_train.shape)

X shape: (352, 1)
y shape: (352, 1)


In [6]:
print(f"[Train] تعداد نمونه‌ها: {X_train.shape[0]}")
print(f"[Train] بازه‌ی HorsePower: [{X_train.min():.3f}, {X_train.max():.3f}]  (داده از قبل نرمالایز شده)")

[Train] تعداد نمونه‌ها: 352
[Train] بازه‌ی HorsePower: [0.000, 0.973]  (داده از قبل نرمالایز شده)


In [7]:
# ساخت Feature Mapping برای رگرسیون غیرخطی
def polynomial_feature_mapping(X, degree):
     # تعداد نمونه‌ها
    n = X.shape[0]
    # ساخت ماتریس ویژگی
    # تعداد ستون‌ها برابر degree + 1 است
    Phi = np.ones((n, degree + 1))
    # ساخت x، x^2، ...، x^degree
    for j in range(1, degree + 1):
        Phi[:, j] = X[:, 0] ** j
    return Phi

In [8]:
# درجه چندجمله‌ای
degree = 3

Phi_train = polynomial_feature_mapping(X_train, degree)

print(f"[Vectorize] درجه‌ی چندجمله‌ای استفاده‌شده: {degree}")
print(f"[Vectorize] شکل ماتریس ویژگی وکتورایزشده Phi_train: {Phi_train.shape}")

[Vectorize] درجه‌ی چندجمله‌ای استفاده‌شده: 3
[Vectorize] شکل ماتریس ویژگی وکتورایزشده Phi_train: (352, 4)


In [29]:
# برای تکرارپذیری این کار رو میکنیم
np.random.seed(12)

w_init = np.random.randn(degree + 1, 1)

print("Initial weights:")
print(w_init)

Initial weights:
[[ 0.47298583]
 [-0.68142588]
 [ 0.2424395 ]
 [-1.70073563]]


In [17]:
# چند مقدار مختلف برای Learning Rate قرار میدم تا نفر سوم بتونه آزمایش کنه
learning_rates = [0.5, 0.1, 0.01]


# نفر سوم ترکیب‌های مختلف این مقادیر را تست میکنه.
epochs_list = [1000, 3000, 5000]

# نفر سوم توان های مختلف این مقادیر را تست میکنه.
d_list = [2,3,4,5]

print(f"[Hyperparams] learning_rates = {learning_rates}")
print(f"[Hyperparams] epochs_list    = {epochs_list}")

[Hyperparams] learning_rates = [0.5, 0.1, 0.01]
[Hyperparams] epochs_list    = [1000, 3000, 5000]


In [16]:
# نمایش اطلاعات
print("Ready for training:")
print("Phi_train:", Phi_train.shape)
print("y_train:", y_train.shape)
print("Initial weights:", w_init.shape)
print("Learning rates:", learning_rates)
print("Epochs:", epochs_list)

Ready for training:
Phi_train: (352, 4)
y_train: (352, 1)
Initial weights: (4, 1)
Learning rates: [0.5, 0.1, 0.01]
Epochs: [1000, 3000, 5000]


<div style="display:inline-block; padding:14px 30px; border-radius:40px 12px 40px 12px; background:#b7d7f0; color:#1e3c5c; font-family:'Segoe UI',sans-serif; font-size:1.6rem; font-weight:500; box-shadow:0 6px 18px rgba(0,0,0,0.06); border:1px solid rgba(255,255,255,0.4);">
    ⭐ Fatemeh Ebrahimi
</div>

Loss Function (MSE) : 

In [24]:
def loss(y,y_hat):
    return np.sum((y - y_hat)**2) / len(y)

Update Weights :

In [25]:
def update(w , gradient , learning_rate):
    return w - learning_rate * gradient

Model :

In [26]:
def regression_model(phi, w):
    return phi @ w

Gradient :

In [27]:
def gradient(phi, y, w):
    n = len(y)
    y_hat = regression_model(phi,w)
    return (2/n) * phi.T @ (y_hat - y)

Train function :

In [28]:
def train(phi, y, w_init, learning_rate, epochs):
    w = w_init.copy()
    losses = []
    for epoch in range(epochs):
        grad = gradient(phi, y, w)
        w = update(w, grad, learning_rate)
        losses.append(loss(y, phi @ w).item())
    return w, losses

Training loop and Getting losses

In [ ]:
np.random.seed(12)
result_list = []
for d in d_list:
    phi = polynomial_feature_mapping(X_train , d)
    for learning_rate in learning_rates:
        for epoch in epochs_list:
            w0 = np.random.randn(d + 1, 1)

            w_final, losses = train(phi, y_train, w0, learning_rate, epoch)
            final_loss = np.sum(losses)/len(losses)
            result_list.append({"degree": d,"learning_rate": learning_rate,"epochs": epoch, "loss" : final_loss, "w_final": w_final})

            print(f"The Model Saved. Epoches : {epoch}\tDegree : {d}\tFinal Loss Value : {final_loss}")

The Model Saved. Epoches : 1000	Degree : 2	Final Loss Value : 0.014604691301706168
The Model Saved. Epoches : 3000	Degree : 2	Final Loss Value : 0.016291747031327256
The Model Saved. Epoches : 5000	Degree : 2	Final Loss Value : 0.014457558129298666
The Model Saved. Epoches : 1000	Degree : 2	Final Loss Value : 0.023348509113184306
The Model Saved. Epoches : 3000	Degree : 2	Final Loss Value : 0.014127921395413957
The Model Saved. Epoches : 5000	Degree : 2	Final Loss Value : 0.019254563938368843
The Model Saved. Epoches : 1000	Degree : 2	Final Loss Value : 0.15426400992581965
The Model Saved. Epoches : 3000	Degree : 2	Final Loss Value : 0.036581715279506676
The Model Saved. Epoches : 5000	Degree : 2	Final Loss Value : 0.05606776295555385
The Model Saved. Epoches : 1000	Degree : 3	Final Loss Value : 0.014544189725381447
The Model Saved. Epoches : 3000	Degree : 3	Final Loss Value : 0.014617836499290528
The Model Saved. Epoches : 5000	Degree : 3	Final Loss Value : 0.01410214829506122
The Mod

Find the Best Model

In [34]:
best_model = min(result_list , key = lambda dict : dict['loss'])

Save Best Model Weights

In [35]:
np.save('./Best_Model',best_model['w_final'])

**The End of Training** 😃🎉✨